In [12]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
model_dtype = torch.bfloat16

In [13]:
from stable_audio_vae import StableAudioVAE

vae = StableAudioVAE(
    config_path="./models/ace-step-vae/config.json",
    checkpoint_path="./models/ace-step-vae/checkpoint.ckpt",
).to(device).to(model_dtype)

In [14]:
from transformers import AutoModel

model = AutoModel.from_pretrained("ACE-Step/acestep-v15-turbo-shift1", trust_remote_code=True,
                                  attn_implementation="eager", dtype=model_dtype).to(device).to(model_dtype)

In [4]:
def cast_input_hook(module, args):
    return tuple(a.to(module.weight.dtype) if isinstance(a, torch.Tensor) else a for a in args)

model.tokenizer.quantizer.project_out.register_forward_pre_hook(cast_input_hook)

In [20]:
text_hidden_states = torch.randn(2, 77, 1024, dtype=model_dtype, device=device)
text_attention_mask = torch.ones(2, 77, dtype=model_dtype, device=device)
lyric_hidden_states = torch.randn(2, 123, 1024, dtype=model_dtype, device=device)
lyric_attention_mask = torch.ones(2, 123, dtype=model_dtype, device=device)
refer_audio_acoustic_hidden_states_packed = torch.randn(3, 750, 64, dtype=model_dtype, device=device)
refer_audio_order_mask = torch.LongTensor([0, 0, 1]).to(device)

base_seconds = 10
frames_per_second = 25
base_seq_len = base_seconds * frames_per_second

hidden_states = torch.randn(2, base_seq_len, 64, dtype=model_dtype, device=device)
attention_mask = torch.ones(2, base_seq_len, dtype=model_dtype, device=device)

pad_start = max(base_seq_len // 2, 1)
attention_mask[0, pad_start:] = 0
chunk_mask = torch.ones(2, base_seq_len, 64, dtype=model_dtype, device=device)
chunk_mask[0, pad_start:] = 0

silence_latent = torch.randn(2, base_seq_len, 64, dtype=model_dtype, device=device)
src_latents = torch.randn(2, base_seq_len, 64, dtype=model_dtype, device=device)
is_covers = torch.Tensor([False])

seconds = 10
infer_steps = 50

seq_len = seconds * frames_per_second

cur_hidden_states = torch.randn(2, seq_len, 64, dtype=model_dtype, device=device)
cur_attention_mask = torch.ones(2, seq_len, dtype=model_dtype, device=device)
cur_chunk_mask = torch.ones(2, seq_len, 64, dtype=model_dtype, device=device)
cur_silence_latent = torch.randn(2, seq_len, 64, dtype=model_dtype, device=device)
cur_src_latents = torch.randn(2, seq_len, 64, dtype=model_dtype, device=device)

print(cur_hidden_states.shape)
print(cur_attention_mask.shape)
print(cur_chunk_mask.shape)
print(cur_silence_latent.shape)
print(cur_src_latents.shape)

torch.Size([2, 250, 64])
torch.Size([2, 250])
torch.Size([2, 250, 64])
torch.Size([2, 250, 64])
torch.Size([2, 250, 64])


In [6]:
outputs = model.generate_audio(
    text_hidden_states=text_hidden_states,
    text_attention_mask=text_attention_mask,
    lyric_hidden_states=lyric_hidden_states,
    lyric_attention_mask=lyric_attention_mask,
    refer_audio_acoustic_hidden_states_packed=refer_audio_acoustic_hidden_states_packed,
    refer_audio_order_mask=refer_audio_order_mask,
    src_latents=cur_src_latents,
    chunk_masks=cur_chunk_mask,
    silence_latent=cur_silence_latent,
    infer_steps=infer_steps,
    is_covers=is_covers,
    seed=1234,
)

C:\repos\audio-diffusion-finetuning\.venv\Lib\site-packages\vector_quantize_pytorch\residual_fsq.py:170: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled = False):


In [19]:
output_latents = outputs['target_latents']
print(output_latents.shape)

torch.Size([2, 250, 64])


In [15]:
output = vae.decode(outputs['target_latents'])

RuntimeError: Given groups=1, weight of size [2048, 64, 7], expected input[2, 250, 64] to have 64 channels, but got 250 channels instead